# Inkling-2-Mini Standalone GGUF Distillation (20-Tensor BF16 Attention LoRA)

This notebook trains a lightweight **20-tensor BF16 attention-only LoRA** (`layers 4..23` `attn.wo_ud`) on `thinkingmachines/Inkling-2-Mini` using our in-house `Qwen3Guard`-filtered hybrid teacher dataset (`teacher_sft.jsonl`).

- **Runtime:** ~15–20 min on Google Colab A100 (40/80 GB) or H100 (80 GB)
- **Output:** `inkling2-mini-cyber-lora.zip` (~10 MB PEFT adapter)
- **Next step (local):** Splice into stock `Inkling-2-Mini-IQ4_NL.gguf` in ~30s via:
  ```bash
  python3 tools/ik2_ablate/apply_lora_gguf.py \
    ~/ai/models/inkling-2-mini/Inkling-2-Mini-IQ4_NL.gguf \
    ./inkling2-mini-cyber-lora \
    ./out-standalone-gguf --scale 1.0 --full
  ```

In [ ]:
!nvidia-smi
!pip install -q -U "transformers>=5.0.0" peft accelerate safetensors datasets huggingface_hub

In [ ]:
# Upload `teacher_sft.jsonl` and `colab_train_lora.py` from your machine (or mount Google Drive)
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

In [ ]:
# Train the 20-tensor BF16 attention-only LoRA (layers 4..23 attn.wo_ud)
!python3 colab_train_lora.py \
  --data teacher_sft.jsonl \
  --outdir inkling2-mini-cyber-lora \
  --layers 4-23 \
  --targets wo_ud \
  --rank 16 \
  --alpha 32 \
  --epochs 2.0 \
  --lr 1.5e-4 \
  --grad-accum 8 \
  --max-seq-len 2560

In [ ]:
# Download the ~10 MB adapter zip to splice locally into stock Inkling-2-Mini-IQ4_NL.gguf
from google.colab import files
files.download("inkling2-mini-cyber-lora.zip")